# 13 · Ethics, Fairness & Accountability

**Goal.** Check the model is fair across platforms, protects privacy, and is framed responsibly.

**Why it matters.** A mental-health tool carries real risk. This notebook states plainly what the system
**is and is not** (a screening aid, never a diagnosis), checks per-platform fairness, and confirms no
usernames are stored.

**Output:** `data/stage4_explain/ethics_accountability.json` + the screening-aid disclaimer.

## 0. Environment

In [1]:
import os, sys, json, logging
from pathlib import Path
from collections import Counter, defaultdict

for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np
def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False
assert _try("sklearn"), "notebook 13 needs sklearn for the parity metrics"
HAS_MPL = _try("matplotlib")

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("ethics")
PART_VERSION = "1.0"
TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}
MIN_SUPPORT = 5   # minimum true-class count on a platform to report a per-tier recall (else n/a)

## 1. Load predictions + platform labels + Objective 3 confusion (frozen TEST, P3)

In [2]:
def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage4_explain").exists() or (d / "data" / "stage3_model").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
MODEL_DIR = PROJECT_ROOT / "data" / "stage3_model"
SEM_DIR   = PROJECT_ROOT / "data" / "stage2_semantic"
XAI_DIR   = PROJECT_ROOT / "data" / "stage4_explain"
XAI_DIR.mkdir(parents=True, exist_ok=True)

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

test_rows = load_jsonl(SEM_DIR / "corpus_preprocessed_test.jsonl")
assert set(r["platform"] for r in test_rows) <= {"reddit", "twitter"}, "TEST platform leak!"
plat = {r["record_id"]: r["platform"] for r in test_rows}
# privacy check: anonymised IDs present, no raw username/author field persisted
anon_ok = all("anonymized_user_id" in r for r in test_rows)
raw_user_leak = any(k in test_rows[0] for k in ("author", "username", "user_name", "screen_name"))

preds = load_jsonl(MODEL_DIR / "predictions_test.jsonl")
clf = json.loads((MODEL_DIR / "classification_report.json").read_text(encoding="utf-8"))
CONFUSION = clf["test_metrics"]["confusion_matrix"]
log.info("TEST: %d rows | platforms=%s | anon_ids=%s raw_user_leak=%s",
         len(preds), dict(Counter(plat.values())), anon_ok, raw_user_leak)
assert anon_ok and not raw_user_leak, "privacy invariant violated (raw username persisted / missing anon id)"


2026-08-08 20:26:47,429 | INFO | TEST: 2768 rows | platforms={'reddit': 1302, 'twitter': 1466} | anon_ids=True raw_user_leak=False


## 2. Automated fairness probe — per-platform performance parity

Split the frozen-TEST predictions by platform and compute per-platform accuracy / macro-F1 / MCC and per-Tier
recall. **Caveat (stated, not hidden):** Reddit and Twitter have **different class priors**, so raw gaps partly
reflect dataset *composition*; the **per-Tier recall** view controls for tier (reported where true-class
support ≥ 5).

In [3]:
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, recall_score

by_plat = defaultdict(lambda: {"y": [], "p": []})
for r in preds:
    pl = plat.get(r["record_id"])
    by_plat[pl]["y"].append(int(r["tier_true"]))
    by_plat[pl]["p"].append(int(r["tier_pred"]))

def platform_metrics(y, p):
    y = np.array(y); p = np.array(p)
    prior = {TIER_NAMES[t]: int((y == t).sum()) for t in range(4)}
    rec = recall_score(y, p, labels=[0,1,2,3], average=None, zero_division=0)
    per_tier_recall = {}
    for t in range(4):
        n_true = int((y == t).sum())
        per_tier_recall[TIER_NAMES[t]] = round(float(rec[t]), 4) if n_true >= MIN_SUPPORT else None
    return {
        "n": int(len(y)), "class_prior": prior,
        "accuracy": round(float(accuracy_score(y, p)), 4),
        "macro_f1": round(float(f1_score(y, p, average="macro", zero_division=0)), 4),
        "mcc": round(float(matthews_corrcoef(y, p)), 4) if len(set(y)) > 1 else None,
        "per_tier_recall": per_tier_recall,
    }

parity = {pl: platform_metrics(d["y"], d["p"]) for pl, d in by_plat.items()}

# disparity summary (overall + per-tier where both platforms have support)
plats = sorted(parity)
def gap(metric_getter):
    vals = [metric_getter(parity[pl]) for pl in plats]
    vals = [v for v in vals if v is not None]
    return round(max(vals) - min(vals), 4) if len(vals) == 2 else None

per_tier_recall_gap = {}
for t in range(4):
    nm = TIER_NAMES[t]
    rr = [parity[pl]["per_tier_recall"][nm] for pl in plats]
    per_tier_recall_gap[nm] = round(abs(rr[0] - rr[1]), 4) if all(v is not None for v in rr) else None

parity_summary = {
    "platforms": plats,
    "macro_f1_gap": gap(lambda m: m["macro_f1"]),
    "accuracy_gap": gap(lambda m: m["accuracy"]),
    "per_tier_recall_gap": per_tier_recall_gap,
    "class_prior_differs": True,
    "confound_note": ("Reddit and Twitter have markedly different class priors (Reddit ~90% Neutral; Twitter "
                      "balanced across Neutral/Stress/Moderate), so raw macro-F1/accuracy gaps partly reflect "
                      "dataset composition, NOT solely model bias. The per-Tier recall view (controlling for tier) "
                      f"is the fairer parity lens; it is reported only where true-class support >= {MIN_SUPPORT}."),
}
print("=== PER-PLATFORM PARITY (frozen TEST) ===")
for pl in plats:
    m = parity[pl]
    print(f"  {pl:8} n={m['n']:4}  acc={m['accuracy']:.3f}  macroF1={m['macro_f1']:.3f}  MCC={m['mcc']}  "
          f"prior={m['class_prior']}")
    print(f"           per-tier recall: " +
          " ".join(f"{k}={v}" for k, v in m["per_tier_recall"].items()))
print("  macro-F1 gap:", parity_summary["macro_f1_gap"], "| per-tier recall gap:", per_tier_recall_gap)

=== PER-PLATFORM PARITY (frozen TEST) ===
  reddit   n=1302  acc=0.909  macroF1=0.467  MCC=0.4302  prior={'Neutral': 1189, 'Stress': 24, 'Moderate': 67, 'Crisis': 22}
           per-tier recall: Neutral=0.9588 Stress=0.125 Moderate=0.5522 Crisis=0.1818
  twitter  n=1466  acc=0.518  macroF1=0.406  MCC=0.3763  prior={'Neutral': 392, 'Stress': 521, 'Moderate': 529, 'Crisis': 24}
           per-tier recall: Neutral=0.8954 Stress=0.3916 Moderate=0.3856 Crisis=0.0417
  macro-F1 gap: 0.0615 | per-tier recall gap: {'Neutral': 0.0634, 'Stress': 0.2666, 'Moderate': 0.1666, 'Crisis': 0.1401}


## 3. Per-Tier limitations from the Objective 3 confusion matrix

The honest failure modes a deployment must account for, read straight off the confusion matrix.

In [4]:
cm = np.array(CONFUSION)
tier_limits = {}
for t in range(4):
    support = int(cm[t].sum())
    recall = round(float(cm[t, t] / cm[t].sum()), 4) if cm[t].sum() else None
    # where do true-t posts go when misclassified?
    misroute = {TIER_NAMES[j]: int(cm[t, j]) for j in range(4) if j != t and cm[t, j] > 0}
    tier_limits[TIER_NAMES[t]] = {"support": support, "recall": recall, "misrouted_to": misroute}

CRISIS_RECALL = tier_limits["Crisis"]["recall"]
limitations = [
    f"Crisis (Tier-3) recall = {CRISIS_RECALL} on {tier_limits['Crisis']['support']} test posts — the model "
    "CANNOT reliably detect crisis at pilot scale; it must NOT be used for crisis triage in its current form.",
    "Neutral-bias: distress posts are frequently misrouted to Neutral (false negatives) — 'Neutral' must NOT be "
    "read as 'safe/clear'; the cost of a missed distress case is asymmetric and is the priority for scale-up.",
    "Pilot-scale training (1,971 TRAIN rows) — all numbers are architecture/machinery-validating, not deployable; "
    "publishable performance requires the deferred data scale-up + GPU fine-tuning.",
    "No demographic labels were collected (privacy by design) — a demographic fairness audit is NOT possible and "
    "is named as future work; only platform parity is computable here.",
    "Explanations are of a pilot-scale model — trigger CONTENT sharpens at scale (Objective 4 caveat).",
]
for nm, v in tier_limits.items():
    log.info("  %-9s recall=%s support=%d misrouted=%s", nm, v["recall"], v["support"], v["misrouted_to"])

2026-08-08 20:26:47,620 | INFO |   Neutral   recall=0.9431 support=1581 misrouted={'Stress': 20, 'Moderate': 63, 'Crisis': 7}


2026-08-08 20:26:47,620 | INFO |   Stress    recall=0.3798 support=545 misrouted={'Neutral': 294, 'Moderate': 40, 'Crisis': 4}


2026-08-08 20:26:47,620 | INFO |   Moderate  recall=0.4044 support=596 misrouted={'Neutral': 333, 'Stress': 9, 'Crisis': 13}


2026-08-08 20:26:47,622 | INFO |   Crisis    recall=0.1087 support=46 misrouted={'Neutral': 20, 'Stress': 4, 'Moderate': 17}


## 4. The ethics & accountability statement — incl. the screening-aid disclaimer

The non-negotiable text carried verbatim into the paper. Note the explicit **P1-vs-deployment** distinction.

In [5]:
DISCLAIMER = (
    "SCREENING AID, NOT A DIAGNOSTIC TOOL. This system produces tier-level risk SIGNALS to help prioritise "
    "human attention across large volumes of multilingual social-media text. Its outputs are NOT clinical "
    "diagnoses and carry no clinical authority. Every flagged case REQUIRES review by a qualified human before "
    "any action is taken. The system MUST NOT be used for autonomous decisions affecting individuals."
)

ethics = {
    "part": "stage4_part4.3_ethics_accountability", "version": PART_VERSION,
    "principle": "P1 (automated methodology); P3 (frozen TEST);  ethics/accountability/disclaimer",
    "screening_aid_disclaimer": DISCLAIMER,
    "intended_use": [
        "Research into cross-platform, multilingual (Hinglish) mental-health signal detection.",
        "Prioritisation/triage SUPPORT that routes posts to human reviewers — never a substitute for them.",
    ],
    "out_of_scope_and_misuse": [
        "Clinical diagnosis or any individual clinical labelling.",
        "Automated or involuntary intervention, or any high-stakes decision without human review.",
        "Surveillance, individual targeting, or attempts to de-anonymise users.",
        "Crisis triage in the current pilot-scale form (Tier-3 recall is 0).",
    ],
    "human_oversight_vs_P1": (
        "P1 ('fully automatic') governs the RESEARCH METHODOLOGY — labelling is automated weak/distant "
        "supervision because no clinical experts were available. It does NOT license unsupervised "
        "DEPLOYMENT: a deployed system requires mandatory human review (the disclaimer). The two are "
        "complementary — automated to BUILD the evidence, human-overseen to ACT on it."
    ),
    "privacy_posture": {
        "user_ids": "Salted SHA-256 anonymised IDs only; raw usernames are NEVER persisted.",
        "salt": "A stable secret salt (kept out of all artefacts/logs) so IDs are consistent yet non-reversible.",
        "anon_ids_present": bool(anon_ok), "raw_username_leak": bool(raw_user_leak),
        "demographics": "No demographic attributes are collected -> no demographic fairness audit is possible "
                        "(named limitation, not faked).",
    },
    "fairness_probe": {"per_platform_parity": parity, "summary": parity_summary},
    "per_tier_limitations": tier_limits,
    "limitations": limitations,
    "accountability": (
        "Outputs are explainable (notebook 11 multi-method attributions; notebook 12 expert-free clinical face-validity) "
        "and auditable (versioned lexicon/anchor/translit tables; seeded, reproducible runs). The unsupportable "
        "'85% expert-alignment' claim has been RETIRED (notebook 12); a clinician kappa-study is named future work."
    ),
}
print("=== SCREENING-AID DISCLAIMER (verbatim into the paper) ===")
print(DISCLAIMER)
print("\n=== PRIVACY ===")
print(" anon IDs present:", anon_ok, "| raw username leak:", raw_user_leak, "| demographics: none collected")

=== SCREENING-AID DISCLAIMER (verbatim into the paper) ===
SCREENING AID, NOT A DIAGNOSTIC TOOL. This system produces tier-level risk SIGNALS to help prioritise human attention across large volumes of multilingual social-media text. Its outputs are NOT clinical diagnoses and carry no clinical authority. Every flagged case REQUIRES review by a qualified human before any action is taken. The system MUST NOT be used for autonomous decisions affecting individuals.

=== PRIVACY ===
 anon IDs present: True | raw username leak: False | demographics: none collected


## 5. Persist ethics_accountability.json

In [6]:
(XAI_DIR / "ethics_accountability.json").write_text(json.dumps(ethics, indent=2, ensure_ascii=False), encoding="utf-8")
log.info("wrote ethics_accountability.json")

2026-08-08 20:26:47,636 | INFO | wrote ethics_accountability.json


## 6. Figure — per-platform parity (metrics + per-tier recall)

In [7]:
if HAS_MPL:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plats = sorted(parity)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    # (a) overall metrics per platform
    metrics = ["accuracy", "macro_f1", "mcc"]
    x = np.arange(len(metrics)); width = 0.36
    for i, pl in enumerate(plats):
        vals = [parity[pl][m] if parity[pl][m] is not None else 0 for m in metrics]
        ax1.bar(x + (i - 0.5) * width, vals, width, label="%s (n=%d)" % (pl, parity[pl]["n"]))
    ax1.set_xticks(x); ax1.set_xticklabels(["accuracy", "macro-F1", "MCC"], fontsize=9)
    ax1.set_title("Per-platform metrics (priors differ — see note)", fontsize=10)
    ax1.legend(fontsize=8); ax1.axhline(0, color="k", lw=0.6)
    # (b) per-tier recall by platform (the fairer, tier-controlled view)
    tiers = [TIER_NAMES[t] for t in range(4)]
    xt = np.arange(len(tiers))
    for i, pl in enumerate(plats):
        vals = [parity[pl]["per_tier_recall"][nm] for nm in tiers]
        vals_plot = [v if v is not None else 0 for v in vals]
        bars = ax2.bar(xt + (i - 0.5) * width, vals_plot, width, label=pl)
        for b, v in zip(bars, vals):
            if v is None:
                ax2.text(b.get_x() + b.get_width()/2, 0.02, "n/a", ha="center", fontsize=7, rotation=90)
    ax2.set_xticks(xt); ax2.set_xticklabels(tiers, fontsize=9); ax2.set_ylim(0, 1.05)
    ax2.set_ylabel("recall"); ax2.set_title("Per-Tier recall by platform (support≥%d)" % MIN_SUPPORT, fontsize=10)
    ax2.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(XAI_DIR / "platform_parity.png", dpi=130); plt.close(fig)
    log.info("wrote platform_parity.png")
else:
    log.warning("matplotlib absent — skipped figure")

2026-08-08 20:26:48,255 | INFO | wrote platform_parity.png


## 7. This step is complete when

**notebook 13 is complete when:** the **screening-aid / human-review disclaimer** is written; the **automated
per-platform parity** probe (per-platform metrics + per-Tier recall, with the class-prior confound stated) is
computed on the frozen TEST; the **per-Tier limitations** are drawn from the Objective 3 confusion matrix; the
**privacy** posture (salted IDs, no raw usernames, no demographics) is restated and asserted; demographic parity
is **named as a limitation** (not faked). `ethics_accountability.json` + `platform_parity.png` produced.
**No human / no expert in the methodology; P3 frozen-TEST; deployment requires human oversight.**

In [8]:
dod = all([
    (XAI_DIR / "ethics_accountability.json").exists(),
    bool(DISCLAIMER),
    "per_platform_parity" in ethics["fairness_probe"],
    anon_ok and not raw_user_leak,
    len(limitations) >= 4,
])
print("=" * 64); print("Objective 4 · notebook 13 — ETHICS & ACCOUNTABILITY CARD"); print("=" * 64)
print("disclaimer     : screening-aid, NOT diagnostic; human review required (verbatim persisted)")
print("fairness probe : per-platform parity — macro-F1 gap = %s (priors differ; per-tier view reported)" %
      parity_summary["macro_f1_gap"])
print("safety         : Crisis recall = %s -> NOT for crisis triage; Neutral-bias -> 'Neutral' != 'safe'" % CRISIS_RECALL)
print("privacy        : salted SHA-256 IDs, no raw usernames (asserted); no demographics -> demo-parity = future work")
print("P1 vs deploy   : automated METHODOLOGY != unsupervised DEPLOYMENT (human review mandatory)")
print("outputs        : ethics_accountability.json, platform_parity.png")
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE", "(accountability layer; deployment claims await scale-up)")

Objective 4 · notebook 13 — ETHICS & ACCOUNTABILITY CARD
disclaimer     : screening-aid, NOT diagnostic; human review required (verbatim persisted)
fairness probe : per-platform parity — macro-F1 gap = 0.0615 (priors differ; per-tier view reported)
safety         : Crisis recall = 0.1087 -> NOT for crisis triage; Neutral-bias -> 'Neutral' != 'safe'
privacy        : salted SHA-256 IDs, no raw usernames (asserted); no demographics -> demo-parity = future work
P1 vs deploy   : automated METHODOLOGY != unsupervised DEPLOYMENT (human review mandatory)
outputs        : ethics_accountability.json, platform_parity.png

Completion check: PASS (accountability layer; deployment claims await scale-up)
